# Model Training
### The setting in this code:
- Data
  - Shuffled & preprocessed Medsynth (created by "dial2note_dataset_creation.ipynb")
  - the number of test dataset: 500
  - the number of eval dataset: 500
  - the number of train dataset: the rest
- Model
  - base model: Qwen3.5-2B
  - new model name (after training): qwen3.5-2B_SFT
  - checkpoint path: "./qwen3.5-2B_SFT_med"
- packing
  - True
  

In [ ]:
# The path for the dataset for training 
shuffled_dataset_path = "/kaggle/working/medsynth_preprocessed_shuffled"
base_model_name = "Qwen/Qwen3.5-2B"
checkpoint_path = "./qwen3.5-2B_SFT_med"
new_model_name = "qwen3.5-2B_SFT"

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install unsloth

In [ ]:
import os
# before import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"    # only if you are using more than 1 GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
import unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq
from datasets import load_from_disk

In [ ]:
# 1. model loading
max_seq_length = 2048
model, processor = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True, # for QLoRA
)
tokenizer = processor.tokenizer # only for text

# 2. LoRA config
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Unsloth is optimized the best when it's 0
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

def format_chatml(example):
    # Instruction
    instruction = (
        "Task: Convert the medical dialogue into a professional SOAP note.\n"
        "Guidelines:\n"
        "1. STRUCTURE:\n"
        " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
        " - Objective: Observable and measurable findings from physical examination.\n"
        " - Assessment: Clinical diagnosis and medical reasoning.\n"
        " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
        "2. CONSTRAINTS:\n"
        " - Use formal clinical terminology.\n"
        " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
        " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
    )
    dialogue = example.get("Dialogue", "")
    note = example.get(" Note", "")

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": f"Dialogue:\n{dialogue}"},
        {"role": "assistant", "content": note}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# 3. Dataset preparation
shuffled_ds = load_from_disk(shuffled_dataset_path)
total_count = len(shuffled_ds)
formatted_ds = shuffled_ds.map(format_chatml, remove_columns=shuffled_ds.column_names)

train_ds = formatted_ds.select(range(total_count - 1000))
eval_ds = formatted_ds.select(range(total_count - 1000, total_count - 500))

print(f"# Train data: {len(train_ds)}")
print(f"# Eval data: {len(eval_ds)}")
print("Trainig and evaluation dataset ready")

# # 4. data collator: for making the data neat
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,      
#     padding=True
# )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

Filter:   0%|          | 0/10240 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10238 [00:00<?, ? examples/s]

중복 제거 전: 10238개 -> 중복 제거 후: 10033개


Map:   0%|          | 0/9033 [00:00<?, ? examples/s]

학습(Train) 데이터 개수: 8533
검증(Eval) 데이터 개수: 500


In [ ]:
# 5. Training
training_args = TrainingArguments(
    output_dir=checkpoint_path,
    max_steps = -1,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    warmup_steps=5,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=100,
    optim="adamw_8bit", # memory saving optimizer
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    
    # saving!
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    average_tokens_across_devices=False
)

# for testing with a small subset of dataset
# training_args = TrainingArguments(
#     output_dir="./qwen_9_med",
#     per_device_train_batch_size=32,
#     gradient_accumulation_steps=1,
#     gradient_checkpointing=True,
#     warmup_steps=5,
#     max_steps=-1, # 전체 에폭 학습 시 -1
#     num_train_epochs=2,
#     learning_rate=2e-4,
#     fp16=True,
#     bf16=False,
#     logging_steps=10,
#     optim="adamw_8bit", # 메모리 절약형 옵티마이저
#     weight_decay=0.01,
#     lr_scheduler_type="cosine",
#     seed=42,

#     save_strategy="no",
#     eval_strategy="no",
#     load_best_model_at_end=False,
#     metric_for_best_model="eval_loss",
#     average_tokens_across_devices=False
# )

# 5. Trainer config (SFTTrainer)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    args = training_args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/8533 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,533 | Num Epochs = 2 | Total steps = 534
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.886712,0.808766
200,0.806755,0.821243
300,0.949757,1.577396
400,5.098331,7.490182


TrainOutput(global_step=400, training_loss=1.9353887939453125, metrics={'train_runtime': 25562.1319, 'train_samples_per_second': 0.668, 'train_steps_per_second': 0.021, 'total_flos': 1.3461847599100723e+17, 'train_loss': 1.9353887939453125, 'epoch': 1.4981273408239701})

In [ ]:
# 6. Model saving
save_path = "./" + new_model_name
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model : {new_model_name} saved!")

Model : qwen_9 saved!
